<a href="https://colab.research.google.com/github/acm-research-f26/Circuitree/blob/Jotish-Chowdary-Boddu/all_circuits_variants.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get -qq update
!apt-get -qq install -y yosys
!which yosys-abc

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package at-spi2-common.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../00-at-spi2-common_2.52.0-1build1_all.deb ...
Unpacking at-spi2-common (2.52.0-1build1) ...
Selecting previously unselected package libatspi2.0-0t64:amd64.
Preparing to unpack .../01-libatspi2.0-0t64_2.52.0-1build1_amd64.deb ...
Unpacking libatspi2.0-0t64:amd64 (2.52.0-1build1) ...
Selecting previously unselected package libxtst6:amd64.
Preparing to unpack .../02-libxtst6_2%3a1.2.3-1.1build1_amd64.deb ...
Unpacking libxtst6:amd64 (2:1.2.3-1.1build1) ...
Selecting previously unselected package session-migration.
Preparing to unpack .../03-session-migration_0.3.9build1_amd64.deb ...
Unpacking session-migration (0.3.9build1) ...
Selecting previously unse

In [1]:
!cp /*.bench /content/
%cd /content
!ls *.bench

cp: cannot stat '/*.bench': No such file or directory
/content
ls: cannot access '*.bench': No such file or directory


In [11]:
import subprocess, random, re, os, csv

def run_abc(cmd):
    r = subprocess.run(["yosys-abc", "-c", cmd], capture_output=True, text=True)
    return r.stdout + r.stderr

def get_stats(path):
    out = run_abc(f"read_bench {path}; strash; print_stats")
    m = re.search(r"and\s*=\s*(\d+)\s+lev\s*=\s*(\d+)", out)
    return int(m.group(1)), int(m.group(2))

def is_equivalent(a, b):
    out = run_abc(f"cec {a} {b}")
    return "Networks are equivalent" in out

pool = ["balance", "rewrite", "rewrite -z", "refactor", "refactor -z", "resub", "dc2"]

def too_close(stats, seen, min_gap=3):
    for (a, l) in seen:
        if abs(stats[0] - a) < min_gap and abs(stats[1] - l) < min_gap:
            return True
    return False

def make_variants(name, want=10, tries=300, seed=0, min_gap=3):
    random.seed(seed)
    orig = f"{name}.bench"
    base = get_stats(orig)
    seen = [base]
    kept = []
    for t in range(tries):
        steps = random.sample(pool, k=random.randint(2, 4))
        recipe = "strash; " + "; ".join(steps)
        out = f"{name}_v{len(kept)+1}.bench"
        run_abc(f"read_bench {orig}; {recipe}; write_bench -l {out}")
        stats = get_stats(out)
        if too_close(stats, seen, min_gap) or not is_equivalent(orig, out):
            os.remove(out)
            continue
        seen.append(stats)
        kept.append((name, out, recipe, base[0], base[1], stats[0], stats[1]))
        if len(kept) == want:
            break
    print(f"{name}: kept {len(kept)} of {want}")
    return kept

In [10]:
%cd /content
rows = make_variants("c432")

/content
c432: kept 10 of 10


In [12]:
circuits = ["c17", "c432", "c499", "c880", "c1355", "c1908",
            "c2670", "c3540", "c5315", "c6288", "c7552"]

all_rows = []
for c in circuits:
    if os.path.exists(f"{c}.bench"):
        all_rows += make_variants(c)
    else:
        print(f"skipping {c}: file not uploaded")

with open("variants_summary.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["circuit", "variant_file", "recipe", "orig_and", "orig_lev", "var_and", "var_lev"])
    w.writerows(all_rows)
print("done:", len(all_rows), "variants total")

c17: kept 0 of 10
c432: kept 10 of 10
c499: kept 5 of 10
c880: kept 10 of 10
c1355: kept 10 of 10
c1908: kept 10 of 10
c2670: kept 10 of 10
c3540: kept 10 of 10
c5315: kept 10 of 10
c6288: kept 7 of 10
c7552: kept 10 of 10
done: 92 variants total


In [13]:
import glob

def make_variants(name, want=10, tries=300, seed=0, min_gap=3, max_steps=4):
    for f in glob.glob(f"{name}_v*.bench"):   # clear old variants of this circuit
        os.remove(f)
    random.seed(seed)
    orig = f"{name}.bench"
    base = get_stats(orig)
    seen = [base]
    kept = []
    for t in range(tries):
        steps = random.sample(pool, k=random.randint(2, max_steps))
        recipe = "strash; " + "; ".join(steps)
        out = f"{name}_v{len(kept)+1}.bench"
        run_abc(f"read_bench {orig}; {recipe}; write_bench -l {out}")
        stats = get_stats(out)
        if too_close(stats, seen, min_gap) or not is_equivalent(orig, out):
            if os.path.exists(out):
                os.remove(out)
            continue
        seen.append(stats)
        kept.append((name, out, recipe, base[0], base[1], stats[0], stats[1]))
        if len(kept) == want:
            break
    print(f"{name}: kept {len(kept)} of {want}")
    return kept

# drop the old rows for the circuits we're redoing
redo = ["c499", "c6288"]
all_rows = [r for r in all_rows if r[0] not in redo]

for c in redo:
    all_rows += make_variants(c, tries=1000, min_gap=1, max_steps=6, seed=1)

c499: kept 10 of 10
c6288: kept 10 of 10


In [15]:
import pandas as pd

df = pd.DataFrame(all_rows, columns=["circuit","variant_file","recipe","orig_and","orig_lev","var_and","var_lev"])

# re-check every variant with CEC
df["ok"] = [is_equivalent(f"{r.circuit}.bench", r.variant_file) for r in df.itertuples()]
print("all equivalent:", df["ok"].all(), "| total:", len(df))

summary = df.groupby("circuit").agg(
    n=("variant_file", "count"),
    orig_and=("orig_and", "first"),
    min_and=("var_and", "min"),
    max_and=("var_and", "max"),
    min_lev=("var_lev", "min"),
    max_lev=("var_lev", "max"),
)
print(summary)

all equivalent: True | total: 100
          n  orig_and  min_and  max_and  min_lev  max_lev
circuit                                                  
c1355    10       504      386      500       18       25
c1908    10       414      361      397       25       28
c2670    10       717      550      646       17       24
c3540    10      1038      926     1015       33       39
c432     10       209      133      203       23       39
c499     10       400      386      394       18       20
c5315    10      1773     1369     1490       28       37
c6288    10      2337     1869     2321       89      119
c7552    10      2074     1455     1685       24       38
c880     10       327      299      321       21       27


In [14]:
with open("variants_summary.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["circuit", "variant_file", "recipe", "orig_and", "orig_lev", "var_and", "var_lev"])
    w.writerows(all_rows)

!zip -q variants.zip *_v*.bench variants_summary.csv
from google.colab import files
files.download("variants.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>